<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.3/blob/main/01_TDNMR_descriptor_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 01_TDNMR_descriptor_extraction.py
# ============================================================
#
# FINAL PUBLICATION PIPELINE
#
# D2O-swollen CPMG
#       ↓
# non-negative ridge ILT
#       ↓
# seven TD-NMR descriptors
#       ↓
# Fig. 2 source data + publication figure
#
# INPUT
# -----
# D2O-swollen CPMG smoothed CSV
#
# OUTPUT
# ------
# 01_TD_NMR_descriptors.csv
# 02_TD_NMR_ILT_distributions.csv
# 03_Fig2_source_data.xlsx
# Fig2_TDNMR.png
# Fig2_TDNMR.pdf
#
# ZIP
# ---
# 01_TDNMR_descriptor_extraction_output.zip
#
# Fig. 2b DISPLAY SETTINGS
# ------------------------
# - Amplitude ticks = 0.00, 0.15, 0.30
# - Figure size = 1.5 × original
# - Long near-zero ILT tails are hidden
# - Internal ILT valleys remain continuous
# - DICL/TRCL/TECL labels shifted left
#
# IMPORTANT
# ---------
# Display processing does NOT affect:
# - ILT distributions saved to CSV
# - seven TD-NMR descriptors
# - source data
# ============================================================


# ============================================================
# 0. INSTALL / IMPORT
# ============================================================

import sys
import subprocess
import importlib.util

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
}

missing = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )


import shutil
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import lsq_linear
from scipy.signal import find_peaks

warnings.filterwarnings("ignore")


# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

N_T2 = 240
RIDGE_ALPHA = 0.02

NORMALIZE_BY_FIRST_POINT = True

SHORT_MAX_MS = 1.0
MID_MAX_MS = 30.0

T2_MIN_MS = 0.1
T2_MAX_MS = 5000.0

PEAK_RELATIVE_HEIGHT = 0.05
PEAK_DISTANCE = 5

EPS = 1e-12


# ============================================================
# 2. FIGURE SETTINGS
# ============================================================

FIG2_REPRESENTATIVE_BASE_IDS = [
    "AMPS_HMA_TECL",
    "AMPS_HMA_TRCL",
    "AMPS_HMA_DICL",
]

CROSSLINKER_ORDER = [
    "DICL",
    "TRCL",
    "TECL",
]

COLORS = {
    "DICL": "#0B6623",
    "TRCL": "#43A047",
    "TECL": "#9AD58A",
}


# ============================================================
# 3. FIG. 2b DISPLAY SETTINGS
# ============================================================

FIG2B_LINEWIDTH = 0.45
FIG2B_ALPHA = 0.95

FIG2B_ELEV = 18
FIG2B_AZIM = -63

FIG2B_BOX_ASPECT = (
    2.3,
    1.25,
    0.85,
)

# Amplitude axis
FIG2B_Z_MIN = 0.0
FIG2B_Z_MAX = 0.30

FIG2B_Z_TICKS = [
    0.00,
    0.15,
    0.30,
]

# Hide only long near-zero tails
FIG2B_DISPLAY_THRESHOLD_REL = 0.01
FIG2B_PADDING_POINTS = 2


# ------------------------------------------------------------
# Crosslinker label position
#
# Previous:
# approximately log10(5000) = 3.70
#
# Revised:
# 3.35 = shifted left to avoid overlap with ILT lines
# ------------------------------------------------------------

FIG2B_CROSSLINKER_LABEL_X = 3.35

# Slightly above the base plane
FIG2B_CROSSLINKER_LABEL_Z = 0.005


# ============================================================
# 4. OUTPUT SETTINGS
# ============================================================

OUTPUT_DIR = Path(
    "01_TDNMR_descriptor_extraction_output"
)

ZIP_PATH = Path(
    "01_TDNMR_descriptor_extraction_output.zip"
)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()


# ============================================================
# 5. UPLOAD INPUT CSV
# ============================================================

try:
    from google.colab import files

    print("=" * 80)
    print("Upload the D2O-swollen CPMG smoothed CSV.")
    print("=" * 80)

    uploaded = files.upload()

    csv_files = [
        Path(name)
        for name in uploaded.keys()
        if name.lower().endswith(".csv")
    ]

except ImportError:

    csv_files = list(
        Path(".").glob("*.csv")
    )


if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV file was found."
    )


preferred = [
    p
    for p in csv_files
    if (
        "d2o" in p.name.lower()
        and "cpmg" in p.name.lower()
        and "smoothed" in p.name.lower()
    )
]


INPUT_FILE = (
    preferred[0]
    if preferred
    else csv_files[0]
)


print("\nInput file:")
print(INPUT_FILE)


# ============================================================
# 6. LOAD INPUT DATA
# ============================================================

df = pd.read_csv(
    INPUT_FILE
)


if df.shape[1] < 3:
    raise ValueError(
        "Input must contain one material-ID column "
        "and multiple CPMG time columns."
    )


material_ids = (
    df.iloc[:, 0]
    .astype(str)
    .str.strip()
    .tolist()
)


# Duplicate check
if len(set(material_ids)) != len(material_ids):

    counts = (
        pd.Series(material_ids)
        .value_counts()
    )

    duplicated = counts[
        counts > 1
    ]

    raise ValueError(
        "Duplicate material IDs were detected:\n"
        + duplicated.to_string()
    )


# Time points
try:

    time_ms = np.asarray(
        [
            float(str(c).strip())
            for c in df.columns[1:]
        ],
        dtype=float,
    )

except Exception as e:

    raise ValueError(
        "All columns after the first column must "
        "contain numeric CPMG times."
    ) from e


signal_matrix = (
    df.iloc[:, 1:]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .to_numpy(dtype=float)
)


if np.isnan(signal_matrix).any():
    raise ValueError(
        "NaN values were detected in the CPMG signal matrix."
    )


if not np.all(
    np.diff(time_ms) > 0
):
    raise ValueError(
        "CPMG time points must be strictly increasing."
    )


print("\nDataset")
print("-------")

print(
    f"Materials   : {len(material_ids)}"
)

print(
    f"Time points : {len(time_ms)}"
)

print(
    "Time range  : "
    f"{time_ms.min():.4f} - "
    f"{time_ms.max():.4f} ms"
)


# ============================================================
# 7. MATERIAL-ID HELPERS
# ============================================================

def base_material_id(material):

    parts = str(material).split("_")

    if (
        len(parts) >= 4
        and parts[-1].isdigit()
    ):
        return "_".join(
            parts[:-1]
        )

    return str(material)


def identify_crosslinker(material):

    base_id = (
        base_material_id(material)
        .upper()
    )

    for cl in CROSSLINKER_ORDER:

        if base_id.endswith(
            "_" + cl
        ):
            return cl

    for cl in CROSSLINKER_ORDER:

        if cl in base_id:
            return cl

    return "Unknown"


crosslinkers = [
    identify_crosslinker(x)
    for x in material_ids
]


unknown = [
    material_ids[i]
    for i, cl in enumerate(crosslinkers)
    if cl == "Unknown"
]


if unknown:

    raise ValueError(
        "Could not identify crosslinker for:\n"
        + "\n".join(unknown)
    )


# ============================================================
# 8. T2 GRID
# ============================================================

t2_grid = np.logspace(
    np.log10(T2_MIN_MS),
    np.log10(T2_MAX_MS),
    N_T2,
)


print("\nILT settings")
print("------------")

print(
    f"T2 grid    : "
    f"{T2_MIN_MS} - {T2_MAX_MS} ms"
)

print(
    f"N_T2       : {N_T2}"
)

print(
    f"Ridge alpha: {RIDGE_ALPHA}"
)


# ============================================================
# 9. ILT KERNEL
# ============================================================

K = np.exp(
    -time_ms[:, None]
    / t2_grid[None, :]
)


K_aug = np.vstack(
    [
        K,
        RIDGE_ALPHA
        * np.eye(N_T2),
    ]
)


# ============================================================
# 10. DECAY PREPROCESSING
# ============================================================

def preprocess_decay(y):

    y = np.asarray(
        y,
        dtype=float,
    ).copy()


    n_tail = max(
        3,
        int(
            np.ceil(
                len(y)
                * 0.05
            )
        ),
    )


    baseline = float(
        np.median(
            y[-n_tail:]
        )
    )


    y_corr = (
        y - baseline
    )


    y_corr = np.clip(
        y_corr,
        0.0,
        None,
    )


    if NORMALIZE_BY_FIRST_POINT:

        norm = float(
            y_corr[0]
        )


        if norm <= EPS:

            norm = float(
                np.max(y_corr)
            )


        if norm <= EPS:

            raise ValueError(
                "Signal became zero after baseline correction."
            )


        y_corr = (
            y_corr
            / norm
        )


    return y_corr


# ============================================================
# 11. NON-NEGATIVE RIDGE ILT
# ============================================================

def run_ilt(y):

    y_aug = np.concatenate(
        [
            y,
            np.zeros(
                N_T2
            ),
        ]
    )


    result = lsq_linear(
        K_aug,
        y_aug,

        bounds=(
            0.0,
            np.inf,
        ),

        method="trf",

        lsmr_tol="auto",

        verbose=0,
    )


    distribution = np.clip(
        result.x,
        0.0,
        None,
    )


    total = float(
        np.sum(
            distribution
        )
    )


    if total <= EPS:

        raise ValueError(
            "ILT distribution has zero intensity."
        )


    distribution = (
        distribution
        / total
    )


    return distribution


# ============================================================
# 12. SEVEN TD-NMR DESCRIPTORS
# ============================================================

def calculate_descriptors(
    distribution
):

    p = np.asarray(
        distribution,
        dtype=float,
    )


    p = (
        p
        / np.sum(p)
    )


    # --------------------------------------------------------
    # 1. Short fraction
    # 2. Mid fraction
    # 3. Long fraction
    # --------------------------------------------------------

    short_mask = (
        t2_grid
        < SHORT_MAX_MS
    )


    mid_mask = (
        (t2_grid >= SHORT_MAX_MS)
        &
        (t2_grid < MID_MAX_MS)
    )


    long_mask = (
        t2_grid
        >= MID_MAX_MS
    )


    short_fraction = float(
        np.sum(
            p[
                short_mask
            ]
        )
    )


    mid_fraction = float(
        np.sum(
            p[
                mid_mask
            ]
        )
    )


    long_fraction = float(
        np.sum(
            p[
                long_mask
            ]
        )
    )


    # --------------------------------------------------------
    # 4. Weighted log-mean T2
    # --------------------------------------------------------

    log_t2 = np.log10(
        t2_grid
    )


    weighted_log10_t2 = float(
        np.sum(
            p
            * log_t2
        )
    )


    weighted_logmean_t2_ms = float(
        10
        ** weighted_log10_t2
    )


    # --------------------------------------------------------
    # 5. Distribution width
    # --------------------------------------------------------

    width_log10_t2 = float(
        np.sqrt(
            np.sum(
                p
                *
                (
                    log_t2
                    - weighted_log10_t2
                )
                ** 2
            )
        )
    )


    # --------------------------------------------------------
    # 6. Number of detected peaks
    # --------------------------------------------------------

    max_height = float(
        np.max(p)
    )


    if max_height <= EPS:

        n_peaks = 0

    else:

        peaks, _ = find_peaks(
            p,

            height=(
                PEAK_RELATIVE_HEIGHT
                * max_height
            ),

            distance=(
                PEAK_DISTANCE
            ),
        )


        n_peaks = int(
            len(peaks)
        )


    # --------------------------------------------------------
    # 7. Normalized entropy
    # --------------------------------------------------------

    positive = (
        p[
            p > EPS
        ]
    )


    entropy = float(
        -np.sum(
            positive
            * np.log(
                positive
            )
        )
    )


    entropy_norm = float(
        entropy
        / np.log(
            len(p)
        )
    )


    return {

        "ShortFraction":
            short_fraction,

        "MidFraction":
            mid_fraction,

        "LongFraction":
            long_fraction,

        "Weighted_logmean_T2_ms":
            weighted_logmean_t2_ms,

        "Width_log10T2":
            width_log10_t2,

        "N_detected_peaks":
            n_peaks,

        "Entropy_norm":
            entropy_norm,
    }


# ============================================================
# 13. RUN ILT FOR ALL MATERIALS
# ============================================================

descriptor_rows = []
ilt_rows = []


print("\nRunning ILT...")


for i, material in enumerate(
    material_ids
):

    y_raw = (
        signal_matrix[i]
    )


    y_processed = (
        preprocess_decay(
            y_raw
        )
    )


    distribution = (
        run_ilt(
            y_processed
        )
    )


    desc = (
        calculate_descriptors(
            distribution
        )
    )


    crosslinker = (
        identify_crosslinker(
            material
        )
    )


    descriptor_rows.append(
        {

            "Canonical_ID":
                material,

            "Base_ID":
                base_material_id(
                    material
                ),

            "Crosslinker":
                crosslinker,

            "ShortFraction":
                desc[
                    "ShortFraction"
                ],

            "MidFraction":
                desc[
                    "MidFraction"
                ],

            "LongFraction":
                desc[
                    "LongFraction"
                ],

            "Weighted_logmean_T2_ms":
                desc[
                    "Weighted_logmean_T2_ms"
                ],

            "Width_log10T2":
                desc[
                    "Width_log10T2"
                ],

            "N_detected_peaks":
                desc[
                    "N_detected_peaks"
                ],

            "Entropy_norm":
                desc[
                    "Entropy_norm"
                ],
        }
    )


    for t2, intensity in zip(
        t2_grid,
        distribution,
    ):

        ilt_rows.append(
            {

                "Canonical_ID":
                    material,

                "Base_ID":
                    base_material_id(
                        material
                    ),

                "Crosslinker":
                    crosslinker,

                "T2_ms":
                    float(t2),

                "Relative_Intensity":
                    float(intensity),
            }
        )


    print(
        f"{i+1:02d}/"
        f"{len(material_ids)}  "
        f"{material}"
    )


descriptor_df = pd.DataFrame(
    descriptor_rows
)


ilt_df = pd.DataFrame(
    ilt_rows
)


# ============================================================
# 14. VALIDATION
# ============================================================

SEVEN_DESCRIPTORS = [

    "ShortFraction",

    "MidFraction",

    "LongFraction",

    "Weighted_logmean_T2_ms",

    "Width_log10T2",

    "N_detected_peaks",

    "Entropy_norm",
]


if descriptor_df[
    SEVEN_DESCRIPTORS
].isna().any().any():

    raise ValueError(
        "NaN detected in TD-NMR descriptors."
    )


fraction_sum = (

    descriptor_df[
        "ShortFraction"
    ]

    +

    descriptor_df[
        "MidFraction"
    ]

    +

    descriptor_df[
        "LongFraction"
    ]
)


if not np.allclose(
    fraction_sum,
    1.0,
    atol=1e-6,
):

    raise ValueError(
        "Short + Mid + Long fractions do not sum to 1."
    )


# ============================================================
# 15. SAVE TD-NMR DESCRIPTORS
# ============================================================

DESCRIPTOR_PATH = (
    OUTPUT_DIR
    / "01_TD_NMR_descriptors.csv"
)


descriptor_df.to_csv(
    DESCRIPTOR_PATH,
    index=False,
)


# ============================================================
# 16. SAVE FULL ILT DISTRIBUTIONS
# ============================================================

ILT_PATH = (
    OUTPUT_DIR
    / "02_TD_NMR_ILT_distributions.csv"
)


ilt_df.to_csv(
    ILT_PATH,
    index=False,
)


# ============================================================
# 17. FIG. 2a SOURCE DATA
# ============================================================

fig2a_rows = []


for base_id in (
    FIG2_REPRESENTATIVE_BASE_IDS
):

    matching_indices = [

        i

        for i, material
        in enumerate(
            material_ids
        )

        if (
            base_material_id(
                material
            )
            == base_id
        )
    ]


    if len(
        matching_indices
    ) == 0:

        raise ValueError(
            f"Representative sample not found: "
            f"{base_id}"
        )


    exact_matches = [

        i

        for i in matching_indices

        if (
            material_ids[i]
            == base_id
        )
    ]


    if exact_matches:

        idx = (
            exact_matches[0]
        )

    else:

        idx = (
            matching_indices[0]
        )


    material = (
        material_ids[idx]
    )


    for t, intensity in zip(
        time_ms,
        signal_matrix[idx],
    ):

        fig2a_rows.append(
            {

                "Canonical_ID":
                    material,

                "Base_ID":
                    base_id,

                "Crosslinker":
                    identify_crosslinker(
                        material
                    ),

                "Time_ms":
                    float(t),

                "Relative_Intensity":
                    float(intensity),
            }
        )


fig2a_df = pd.DataFrame(
    fig2a_rows
)


# ============================================================
# 18. FIG. 2b-e SOURCE DATA
# ============================================================

fig2b_df = (
    ilt_df.copy()
)


fig2c_df = (
    descriptor_df[
        [
            "Canonical_ID",
            "Base_ID",
            "Crosslinker",
            "Weighted_logmean_T2_ms",
        ]
    ]
    .copy()
)


fig2d_df = (
    descriptor_df[
        [
            "Canonical_ID",
            "Base_ID",
            "Crosslinker",
            "Width_log10T2",
        ]
    ]
    .copy()
)


fig2e_df = (
    descriptor_df[
        [
            "Canonical_ID",
            "Base_ID",
            "Crosslinker",
            "LongFraction",
        ]
    ]
    .copy()
)


# ============================================================
# 19. SAVE FIG. 2 SOURCE DATA
# ============================================================

SOURCE_DATA_PATH = (
    OUTPUT_DIR
    / "03_Fig2_source_data.xlsx"
)


with pd.ExcelWriter(
    SOURCE_DATA_PATH,
    engine="openpyxl",
) as writer:

    fig2a_df.to_excel(
        writer,
        sheet_name="Fig2a_CPMG_decay",
        index=False,
    )


    fig2b_df.to_excel(
        writer,
        sheet_name="Fig2b_ILT_distribution",
        index=False,
    )


    fig2c_df.to_excel(
        writer,
        sheet_name="Fig2c_Mobility",
        index=False,
    )


    fig2d_df.to_excel(
        writer,
        sheet_name="Fig2d_Heterogeneity",
        index=False,
    )


    fig2e_df.to_excel(
        writer,
        sheet_name="Fig2e_LongFraction",
        index=False,
    )


# ============================================================
# 20. FIG. 2b DISPLAY REGION
# ============================================================
#
# Removes only long near-zero tails.
#
# IMPORTANT:
# Internal low-intensity points are retained.
# Therefore profiles remain continuous.
# ============================================================

def get_continuous_ilt_display_region(
    t2,
    intensity,
    relative_threshold=FIG2B_DISPLAY_THRESHOLD_REL,
    padding_points=FIG2B_PADDING_POINTS,
):

    t2 = np.asarray(
        t2,
        dtype=float,
    )


    z = np.asarray(
        intensity,
        dtype=float,
    )


    if len(z) == 0:

        return (
            np.array([]),
            np.array([]),
        )


    zmax = float(
        np.nanmax(z)
    )


    if (
        not np.isfinite(zmax)
        or zmax <= EPS
    ):

        return (
            np.array([]),
            np.array([]),
        )


    threshold = (
        relative_threshold
        * zmax
    )


    meaningful = np.where(
        z > threshold
    )[0]


    if len(
        meaningful
    ) == 0:

        return (
            np.array([]),
            np.array([]),
        )


    start = max(
        0,
        int(
            meaningful[0]
        )
        - padding_points,
    )


    end = min(
        len(z),
        int(
            meaningful[-1]
        )
        + padding_points
        + 1,
    )


    return (
        t2[
            start:end
        ],

        z[
            start:end
        ],
    )


# ============================================================
# 21. GENERAL FIGURE HELPERS
# ============================================================

def style_axis(ax):

    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


    ax.tick_params(
        axis="both",
        labelsize=8,
    )


def box_scatter(
    ax,
    dataframe,
    value_column,
    ylabel,
):

    grouped_data = []


    for cl in (
        CROSSLINKER_ORDER
    ):

        values = (
            dataframe.loc[
                dataframe[
                    "Crosslinker"
                ] == cl,
                value_column,
            ]
            .astype(float)
            .to_numpy()
        )


        grouped_data.append(
            values
        )


    positions = np.arange(
        1,
        len(
            CROSSLINKER_ORDER
        )
        + 1,
    )


    bp = ax.boxplot(
        grouped_data,

        positions=positions,

        widths=0.55,

        patch_artist=True,

        showfliers=False,

        medianprops={
            "color": "black",
            "linewidth": 1.0,
        },

        whiskerprops={
            "color": "black",
            "linewidth": 0.8,
        },

        capprops={
            "color": "black",
            "linewidth": 0.8,
        },

        boxprops={
            "edgecolor": "black",
            "linewidth": 0.8,
        },
    )


    for patch, cl in zip(
        bp["boxes"],
        CROSSLINKER_ORDER,
    ):

        patch.set_facecolor(
            COLORS[
                cl
            ]
        )

        patch.set_alpha(
            0.30
        )


    rng = np.random.default_rng(
        42
    )


    for pos, cl, values in zip(
        positions,
        CROSSLINKER_ORDER,
        grouped_data,
    ):

        jitter = rng.normal(
            loc=0.0,
            scale=0.055,
            size=len(values),
        )


        ax.scatter(
            np.full(
                len(values),
                pos,
            )
            + jitter,

            values,

            s=14,

            facecolor=COLORS[
                cl
            ],

            edgecolor="black",

            linewidth=0.25,

            alpha=0.85,

            zorder=3,
        )


    ax.set_xticks(
        positions
    )


    ax.set_xticklabels(
        CROSSLINKER_ORDER,
        fontsize=8,
    )


    ax.set_ylabel(
        ylabel,
        fontsize=8,
    )


    style_axis(
        ax
    )


# ============================================================
# 22. CREATE FIGURE 2
# ============================================================
#
# Original = 10.5 × 6.3
# Revised  = 15.75 × 9.45
#
# Exactly 1.5 × larger.
# ============================================================

fig = plt.figure(
    figsize=(
        15.75,
        9.45,
    )
)


gs = fig.add_gridspec(
    nrows=2,
    ncols=6,

    height_ratios=[
        1.15,
        1.0,
    ],

    hspace=0.48,
    wspace=1.05,
)


# ============================================================
# FIG. 2a
# Representative D2O-swollen CPMG curves
# ============================================================

ax_a = fig.add_subplot(
    gs[
        0,
        0:3,
    ]
)


for base_id in (
    FIG2_REPRESENTATIVE_BASE_IDS
):

    sub = (
        fig2a_df[
            fig2a_df[
                "Base_ID"
            ]
            == base_id
        ]
    )


    cl = (
        sub[
            "Crosslinker"
        ]
        .iloc[0]
    )


    ax_a.plot(
        sub[
            "Time_ms"
        ],

        sub[
            "Relative_Intensity"
        ],

        linewidth=1.25,

        color=COLORS[
            cl
        ],

        label=base_id,
    )


ax_a.set_xlabel(
    "Time (ms)",
    fontsize=8,
)


ax_a.set_ylabel(
    "Relative intensity",
    fontsize=8,
)


ax_a.legend(
    frameon=False,
    fontsize=6.5,
    loc="upper right",
)


style_axis(
    ax_a
)


# ============================================================
# FIG. 2b
# 3D STACKED ILT DISTRIBUTIONS
# ============================================================

ax_b = fig.add_subplot(
    gs[
        0,
        3:6,
    ],

    projection="3d",
)


# ------------------------------------------------------------
# Arrange samples by crosslinker
# ------------------------------------------------------------

ordered_materials = []


for cl in (
    CROSSLINKER_ORDER
):

    members = (
        descriptor_df.loc[
            descriptor_df[
                "Crosslinker"
            ] == cl,
            "Canonical_ID",
        ]
        .tolist()
    )


    ordered_materials.extend(
        members
    )


# ------------------------------------------------------------
# Draw continuous ILT profiles
# ------------------------------------------------------------

for sample_no, material in enumerate(
    ordered_materials,
    start=1,
):

    sub = (
        ilt_df[
            ilt_df[
                "Canonical_ID"
            ]
            == material
        ]
        .sort_values(
            "T2_ms"
        )
    )


    cl = (
        identify_crosslinker(
            material
        )
    )


    t2_full = (
        sub[
            "T2_ms"
        ]
        .to_numpy(
            dtype=float
        )
    )


    z_full = (
        sub[
            "Relative_Intensity"
        ]
        .to_numpy(
            dtype=float
        )
    )


    # Remove long near-zero tails only
    t2_plot, z_plot = (
        get_continuous_ilt_display_region(
            t2_full,
            z_full,
        )
    )


    if len(
        t2_plot
    ) == 0:

        continue


    x_plot = np.log10(
        t2_plot
    )


    y_plot = np.full(
        len(
            x_plot
        ),

        sample_no,

        dtype=float,
    )


    ax_b.plot(
        x_plot,

        y_plot,

        z_plot,

        color=COLORS[
            cl
        ],

        linewidth=(
            FIG2B_LINEWIDTH
        ),

        alpha=(
            FIG2B_ALPHA
        ),

        antialiased=True,

        solid_capstyle="round",

        solid_joinstyle="round",
    )


# ============================================================
# FIG. 2b AXIS LIMITS
# ============================================================

ax_b.set_xlim(
    np.log10(
        T2_MIN_MS
    ),
    np.log10(
        T2_MAX_MS
    ),
)


ax_b.set_ylim(
    1,
    len(
        ordered_materials
    ),
)


ax_b.set_zlim(
    FIG2B_Z_MIN,
    FIG2B_Z_MAX,
)


# ============================================================
# FIG. 2b X TICKS
# ============================================================

x_tick_min = int(
    np.ceil(
        np.log10(
            T2_MIN_MS
        )
    )
)


x_tick_max = int(
    np.floor(
        np.log10(
            T2_MAX_MS
        )
    )
)


x_ticks = np.arange(
    x_tick_min,
    x_tick_max + 1,
)


ax_b.set_xticks(
    x_ticks
)


ax_b.set_xticklabels(
    [
        str(x)
        for x in x_ticks
    ],
    fontsize=7,
)


# ============================================================
# FIG. 2b SAMPLE-NUMBER TICKS
# ============================================================

n_samples = len(
    ordered_materials
)


middle_sample = int(
    np.ceil(
        n_samples / 2
    )
)


ax_b.set_yticks(
    [
        1,
        middle_sample,
        n_samples,
    ]
)


ax_b.set_yticklabels(
    [
        "1",
        str(
            middle_sample
        ),
        str(
            n_samples
        ),
    ],
    fontsize=7,
)


# ============================================================
# FIG. 2b AMPLITUDE TICKS
# ============================================================

ax_b.set_zticks(
    FIG2B_Z_TICKS
)


ax_b.set_zticklabels(
    [
        "0.00",
        "0.15",
        "0.30",
    ],
    fontsize=7,
)


# ============================================================
# FIG. 2b AXIS LABELS
# ============================================================

ax_b.set_xlabel(
    r"$\log_{10}(T_2\ [\mathrm{ms}])$",
    fontsize=9,
    labelpad=7,
)


ax_b.set_ylabel(
    "Sample No.",
    fontsize=9,
    labelpad=7,
)


ax_b.set_zlabel(
    "Amplitude",
    fontsize=9,
    labelpad=5,
)


# ============================================================
# FIG. 2b VIEW
# ============================================================

ax_b.view_init(
    elev=FIG2B_ELEV,
    azim=FIG2B_AZIM,
)


ax_b.set_box_aspect(
    FIG2B_BOX_ASPECT
)


# ============================================================
# FIG. 2b GRID / PANES
# ============================================================

ax_b.grid(
    True
)


try:

    ax_b.xaxis.pane.set_alpha(
        0.03
    )

    ax_b.yaxis.pane.set_alpha(
        0.03
    )

    ax_b.zaxis.pane.set_alpha(
        0.03
    )

except Exception:

    pass


ax_b.tick_params(
    axis="both",
    labelsize=7,
    pad=1,
)


# ============================================================
# FIG. 2b CROSSLINKER LABELS
#
# Revised:
# Labels are shifted LEFT along the log10(T2) direction.
#
# Previous position:
# ~3.70
#
# Current position:
# 3.35
# ============================================================

start = 0


for cl in (
    CROSSLINKER_ORDER
):

    members = (
        descriptor_df.loc[
            descriptor_df[
                "Crosslinker"
            ] == cl,
            "Canonical_ID",
        ]
        .tolist()
    )


    if len(
        members
    ) == 0:

        continue


    center = (
        start
        + (
            len(
                members
            )
            + 1
        )
        / 2
    )


    ax_b.text(
        FIG2B_CROSSLINKER_LABEL_X,

        center,

        FIG2B_CROSSLINKER_LABEL_Z,

        cl,

        color=COLORS[
            cl
        ],

        fontsize=8,

        horizontalalignment="right",

        verticalalignment="center",
    )


    start += len(
        members
    )


# ============================================================
# FIG. 2c
# Mobility-related feature
# ============================================================

ax_c = fig.add_subplot(
    gs[
        1,
        0:2,
    ]
)


box_scatter(
    ax_c,

    fig2c_df,

    "Weighted_logmean_T2_ms",

    r"Weighted mean $T_2$ (ms)",
)


ax_c.set_title(
    "Mobility-related feature",
    fontsize=8,
    pad=5,
)


# ============================================================
# FIG. 2d
# Heterogeneity feature
# ============================================================

ax_d = fig.add_subplot(
    gs[
        1,
        2:4,
    ]
)


box_scatter(
    ax_d,

    fig2d_df,

    "Width_log10T2",

    r"Distribution width in $\log_{10}(T_2)$",
)


ax_d.set_title(
    "Heterogeneity feature",
    fontsize=8,
    pad=5,
)


# ============================================================
# FIG. 2e
# Long-component feature
# ============================================================

ax_e = fig.add_subplot(
    gs[
        1,
        4:6,
    ]
)


box_scatter(
    ax_e,

    fig2e_df,

    "LongFraction",

    r"Long-$T_2$ fraction",
)


ax_e.set_title(
    "Long-component feature",
    fontsize=8,
    pad=5,
)


# ============================================================
# 23. SAVE FIGURE
# ============================================================

FIG_PNG = (
    OUTPUT_DIR
    / "Fig2_TDNMR.png"
)


FIG_PDF = (
    OUTPUT_DIR
    / "Fig2_TDNMR.pdf"
)


plt.savefig(
    FIG_PNG,

    dpi=600,

    bbox_inches="tight",
)


plt.savefig(
    FIG_PDF,

    bbox_inches="tight",
)


plt.close(
    fig
)


# ============================================================
# 24. VERIFY OUTPUTS
# ============================================================

EXPECTED_OUTPUTS = [

    DESCRIPTOR_PATH,

    ILT_PATH,

    SOURCE_DATA_PATH,

    FIG_PNG,

    FIG_PDF,
]


for path in (
    EXPECTED_OUTPUTS
):

    if not path.exists():

        raise FileNotFoundError(
            f"Expected output was not generated: "
            f"{path}"
        )


# ============================================================
# 25. ZIP EXACTLY FIVE FILES
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=(
        zipfile.ZIP_DEFLATED
    ),
) as z:

    for path in (
        EXPECTED_OUTPUTS
    ):

        z.write(
            path,
            arcname=path.name,
        )


# ============================================================
# 26. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "01_TDNMR_descriptor_extraction "
    "completed successfully."
)


print(
    "=" * 80
)


print(
    "\nGenerated files:"
)


for path in (
    EXPECTED_OUTPUTS
):

    print(
        "  -",
        path.name,
    )


print(
    "\nSeven TD-NMR descriptors:"
)


print(
    descriptor_df[
        [
            "Canonical_ID",
            "Crosslinker",
        ]
        + SEVEN_DESCRIPTORS
    ]
    .head()
)


print(
    "\nFig. 2b display settings:"
)


print(
    "  Amplitude range        : 0.00 - 0.30"
)


print(
    "  Amplitude ticks        : 0.00, 0.15, 0.30"
)


print(
    "  Figure size            : 15.75 × 9.45 inch"
)


print(
    "  Near-zero tails        : hidden"
)


print(
    "  Internal valleys       : retained"
)


print(
    "  Crosslinker label X    : "
    f"{FIG2B_CROSSLINKER_LABEL_X}"
)


print(
    "  Crosslinker labels     : shifted left"
)


print(
    "\nIMPORTANT:"
)


print(
    "  Fig. 2b display processing does NOT alter "
    "the ILT distributions or seven descriptors."
)


print(
    "\nZIP:"
)


print(
    ZIP_PATH
)


# ============================================================
# 27. AUTOMATIC DOWNLOAD
# ============================================================

try:

    from google.colab import files

    files.download(
        str(
            ZIP_PATH
        )
    )

except ImportError:

    print(
        "\nNot running in Google Colab. "
        "The ZIP remains in the current directory."
    )

Upload the D2O-swollen CPMG smoothed CSV.


Saving AddD2O_CPMG_T2_Relative_Intensity_smoothed_44copolymers.csv to AddD2O_CPMG_T2_Relative_Intensity_smoothed_44copolymers (6).csv

Input file:
AddD2O_CPMG_T2_Relative_Intensity_smoothed_44copolymers (6).csv

Dataset
-------
Materials   : 43
Time points : 200
Time range  : 4.0211 - 801.8997 ms

ILT settings
------------
T2 grid    : 0.1 - 5000.0 ms
N_T2       : 240
Ridge alpha: 0.02

Running ILT...
01/43  AMPS_HMA_TECL
02/43  pSSA_HMA_TECL
03/43  MEDSAH_HMA_TECL
04/43  VBA_HMA_TECL_1
05/43  NIPAM_HMA_TECL
06/43  HEA_HMA_TECL
07/43  AMPS_HMA_TRCL
08/43  pSSA_HMA_TRCL
09/43  MEDSAH_HMA_TRCL
10/43  VBA_HMA_TRCL
11/43  NIPAM_HMA_TRCL
12/43  AMPS_TFEMA_TECL
13/43  pSSA_TFEMA_TECL
14/43  MEDSAH_TFEMA_TECL_1
15/43  VBA_TFEMA_TECL
16/43  NIPAM_TFEMA_TECL
17/43  HEA_TFEMA_TECL
18/43  AMPS_TFEMA_TRCL
19/43  pSSA_TFEMA_TRCL
20/43  MEDSAH_TFEMA_TRCL
21/43  VBA_TFEMA_TRCL
22/43  NIPAM_TFEMA_TRCL
23/43  HEA_TFEMA_TRCL
24/43  AMPS_HMA_DICL
25/43  pSSA_HMA_DICL
26/43  MEDSAH_HMA_DICL
27/43  VBA_HMA

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>